In [1]:
import os

# Fix TF/protobuf runtime crash ("MessageFactory has no attribute GetPrototype")
# by forcing the pure-Python protobuf implementation (compatible with newer protobuf builds).
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "3"

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_PATH = "../input/petfinder-pawpularity-score"
TRAIN_CSV_PATH = os.path.join(BASE_PATH, "train.csv")
TEST_CSV_PATH = os.path.join(BASE_PATH, "test.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_PATH, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train")
TEST_IMG_DIR = os.path.join(BASE_PATH, "test")

train_csv = pd.read_csv(TRAIN_CSV_PATH)
test_csv = pd.read_csv(TEST_CSV_PATH)
submission = pd.read_csv(SAMPLE_SUB_PATH)

print(train_csv.shape, test_csv.shape, submission.shape)
print(train_csv.columns.tolist())



2026-01-21 13:43:57.449080: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769003037.463251      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769003037.467619      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-21 13:43:57.486715: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

(8920, 14) (992, 13) (992, 2)
['Id', 'Subject Focus', 'Eyes', 'Face', 'Near', 'Action', 'Accessory', 'Group', 'Collage', 'Human', 'Occlusion', 'Info', 'Blur', 'Pawpularity']


In [2]:
train_csv = train_csv.drop_duplicates().reset_index(drop=True)
test_csv = test_csv.drop_duplicates().reset_index(drop=True)

feature_cols = [c for c in train_csv.columns if c not in ["Id", "Pawpularity"]]

assert all(
    c in test_csv.columns for c in feature_cols
), "Mismatch train/test feature columns"



In [3]:
IMG_SIZE = (64, 64)


def load_images_from_ids(ids, img_dir, size=(64, 64)):
    images = []
    missing = 0
    for _id in ids:
        fp = os.path.join(img_dir, f"{_id}.jpg")
        img = cv2.imread(fp)
        if img is None:
            missing += 1
            img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        images.append(img)
    if missing:
        print(
            f"Warning: {missing} images missing in {img_dir}. They were replaced with zeros."
        )
    return np.stack(images, axis=0)


train_ids = train_csv["Id"].values
test_ids = test_csv["Id"].values

train_img = load_images_from_ids(train_ids, TRAIN_IMG_DIR, size=IMG_SIZE)
test_img = load_images_from_ids(test_ids, TEST_IMG_DIR, size=IMG_SIZE)

print("train_img:", train_img.shape, "test_img:", test_img.shape)



train_img: (8920, 64, 64, 3) test_img: (992, 64, 64, 3)


In [4]:
train_csv_data = train_csv.copy()
test_csv_data = test_csv.copy()

train_csv_x = train_csv_data[feature_cols].astype(np.float32)
train_y = train_csv_data["Pawpularity"].astype(np.float32)

test_csv_x = test_csv_data[feature_cols].astype(np.float32)

print(train_csv_x.shape, train_y.shape, test_csv_x.shape)



(8920, 12) (8920,) (992, 12)


In [5]:
csv_input = tf.keras.Input(shape=train_csv_x.shape[1:], name="CSV_Input")
img_input = tf.keras.Input(shape=train_img.shape[1:], name="IMG_Input")

csv_hidden1 = tf.keras.layers.Dense(
    200, activation="elu", kernel_initializer="he_normal", name="CSV_Hidden1"
)(csv_input)
csv_hidden2 = tf.keras.layers.Dense(
    200, activation="elu", kernel_initializer="he_normal", name="CSV_Hidden2"
)(csv_hidden1)
csv_hidden3 = tf.keras.layers.Dense(
    200, activation="elu", kernel_initializer="he_normal", name="CSV_Hidden3"
)(csv_hidden2)
csv_hidden4 = tf.keras.layers.Dense(
    200, activation="elu", kernel_initializer="he_normal", name="CSV_Hidden4"
)(csv_hidden3)
csv_hidden5 = tf.keras.layers.Dense(
    200, activation="elu", kernel_initializer="he_normal", name="CSV_Hidden5"
)(csv_hidden4)
csv_hidden6 = tf.keras.layers.Dense(
    200, activation="elu", kernel_initializer="he_normal", name="CSV_Hidden6"
)(csv_hidden5)
csv_dropout = tf.keras.layers.Dropout(0.5, name="CSV_Dropout")(csv_hidden6)

img_conv1 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv1",
)(img_input)
img_conv2 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv2",
)(img_conv1)
img_pooling1 = tf.keras.layers.MaxPooling2D(4, name="IMG_Max1")(img_conv2)

img_conv3 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv3",
)(img_pooling1)
img_conv4 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv4",
)(img_conv3)
img_pooling2 = tf.keras.layers.MaxPooling2D(4, name="IMG_Max2")(img_conv4)

img_conv5 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv5",
)(img_pooling2)
img_conv6 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv6",
)(img_conv5)
img_pooling3 = tf.keras.layers.MaxPooling2D(3, name="IMG_Max3")(img_conv6)

img_dropout = tf.keras.layers.Dropout(0.5, name="IMG_Dropout")(img_pooling3)
img_conv7 = tf.keras.layers.Conv2D(
    120,
    4,
    padding="same",
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_Conv7",
)(img_dropout)

flatten = tf.keras.layers.Flatten(name="IMG_Flatten")(img_conv7)

img_hidden1 = tf.keras.layers.Dense(
    300,
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_hidden1",
    use_bias=False,
)(flatten)
img_dropout1 = tf.keras.layers.Dropout(0.5, name="IMG_Dropout1")(img_hidden1)

img_hidden2 = tf.keras.layers.Dense(
    300,
    activation="elu",
    kernel_initializer="he_normal",
    name="IMG_hidden2",
    use_bias=False,
)(img_dropout1)
img_dropout2 = tf.keras.layers.Dropout(0.5, name="IMG_Dropout2")(img_hidden2)

csv_output = tf.keras.layers.Dense(1, name="CSV_Output")(csv_dropout)
img_output = tf.keras.layers.Dense(1, name="IMG_Output")(img_dropout2)

model = tf.keras.Model(
    inputs=[csv_input, img_input],
    outputs=[csv_output, img_output],
    name="Pythonash_model",
)
model.summary()



2026-01-21 13:44:45.405199: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1769003085.405670      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


Model: "Pythonash_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ IMG_Input           │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv1 (Conv2D)  │ (None, 64, 64,    │      5,880 │ IMG_Input[0][0]   │
│                     │ 120)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv2 (Conv2D)  │ (None, 64, 64,    │    230,520 │ IMG_Conv1[0][0]   │
│                     │ 120)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Max1            │ (None, 16, 16,    │          0 │ IMG_Conv2[0][0]   │
│ (MaxPooling2D)      │ 120)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv3 (Conv2D)  │ (None, 16, 16,    │    230,520 │ IMG_Max1[0][0]    │
│                     │ 120)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv4 (Conv2D)  │ (None, 16, 16,    │    230,520 │ IMG_Conv3[0][0]   │
│                     │ 120)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Max2            │ (None, 4, 4, 120) │          0 │ IMG_Conv4[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv5 (Conv2D)  │ (None, 4, 4, 120) │    230,520 │ IMG_Max2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv6 (Conv2D)  │ (None, 4, 4, 120) │    230,520 │ IMG_Conv5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CSV_Input           │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Max3            │ (None, 1, 1, 120) │          0 │ IMG_Conv6[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CSV_Hidden1 (Dense) │ (None, 200)       │      2,600 │ CSV_Input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Dropout         │ (None, 1, 1, 120) │          0 │ IMG_Max3[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CSV_Hidden2 (Dense) │ (None, 200)       │     40,200 │ CSV_Hidden1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Conv7 (Conv2D)  │ (None, 1, 1, 120) │    230,520 │ IMG_Dropout[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CSV_Hidden3 (Dense) │ (None, 200)       │     40,200 │ CSV_Hidden2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_Flatten         │ (None, 120)       │          0 │ IMG_Conv7[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CSV_Hidden4 (Dense) │ (None, 200)       │     40,200 │ CSV_Hidden3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ IMG_hidden1 (Dense) │ (None, 300)       │     36,000 │ IMG_Flatten[0][0] │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,719,102 (6.56 MB)

 Trainable params: 1,719,102 (6.56 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
learning_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.002, decay_steps=10000, decay_rate=0.97
)
opt = tf.keras.optimizers.Adam(learning_rate=learning_schedule)

# Fix: for multi-output models, metrics must be list/tuple/dict in Keras 2.18
rmse = tf.keras.metrics.RootMeanSquaredError(name="rmse")
model.compile(
    loss=["mse", "mse"],
    loss_weights=[0.5, 0.5],
    optimizer=opt,
    metrics=[rmse, rmse],
)

epoch_number = 5

ckpt_path = "pythonash_model.keras"
check_1 = tf.keras.callbacks.ModelCheckpoint(
    ckpt_path, save_best_only=True, verbose=2, monitor="val_loss", mode="min"
)

history = model.fit(
    x=[train_csv_x, train_img],
    y=[train_y, train_y],
    epochs=epoch_number,
    validation_split=0.2,
    verbose=2,
    batch_size=100,
    validation_batch_size=100,
    callbacks=[check_1],
)



Epoch 1/5


I0000 00:00:1769003090.337983      62 service.cc:148] XLA service 0x7fa82801bfa0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769003090.338029      62 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-21 13:44:50.397660: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769003090.674752      62 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769003096.495192      62 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_loss improved from inf to 406.46133, saving model to pythonash_model.keras
72/72 - 18s - 248ms/step - CSV_Output_loss: 514.6978 - CSV_Output_rmse: 45.3632 - IMG_Output_loss: 3570.7266 - IMG_Output_rmse: 45.3632 - loss: 2057.8184 - val_CSV_Output_loss: 405.1315 - val_CSV_Output_rmse: 20.1609 - val_IMG_Output_loss: 409.2170 - val_IMG_Output_rmse: 20.1609 - val_loss: 406.4613
Epoch 2/5

Epoch 2: val_loss did not improve from 406.46133
72/72 - 2s - 32ms/step - CSV_Output_loss: 450.4504 - CSV_Output_rmse: 21.2980 - IMG_Output_loss: 454.6136 - IMG_Output_rmse: 21.2980 - loss: 453.6040 - val_CSV_Output_loss: 408.7796 - val_CSV_Output_rmse: 20.1882 - val_IMG_Output_loss: 407.8635 - val_IMG_Output_rmse: 20.1882 - val_loss: 407.5621
Epoch 3/5

Epoch 3: val_loss improved from 406.46133 to 406.26682, saving model to pythonash_model.keras
72/72 - 2s - 33ms/step - CSV_Output_loss: 450.1157 - CSV_Output_rmse: 21.2737 - IMG_Output_loss: 452.6573 - IMG_Output_rmse: 21.2737 - loss: 452.568

In [7]:
# Inference + submission writing
# Robustly load the best checkpoint if it exists, otherwise use the in-memory model.
if os.path.exists(ckpt_path):
    best_model = tf.keras.models.load_model(ckpt_path, compile=False)
else:
    best_model = model

csv_result, img_result = best_model.predict(
    [test_csv_x, test_img], batch_size=100, verbose=0
)

final_pred = 0.5 * csv_result.reshape(-1) + 0.5 * img_result.reshape(-1)
final_pred = np.clip(final_pred, 0.0, 100.0)

sub = pd.DataFrame(
    {"Id": test_csv_data["Id"].values, "Pawpularity": final_pred.astype(np.float32)}
)
sub = sub[["Id", "Pawpularity"]]

out_path = "submission.csv"
sub.to_csv(out_path, index=False)

print(sub.head())
print(f"Saved submission to: {out_path}  rows={len(sub)}  cols={sub.columns.tolist()}")

                                 Id  Pawpularity
0  ee51b99832f1ba868f646df93d2b6b81    38.043869
1  caddfb3f8bff9c4b95dbe022018eea21    39.457878
2  582eeabd4a448a53ebb79995888a4b0b    38.071907
3  afc1ad7f0c5eea880759d09e77f7deee    38.023003
4  d5bdf3446e86ce4ec67ce7a00f1cccc2    38.244003
Saved submission to: submission.csv  rows=992  cols=['Id', 'Pawpularity']
